In [ ]:
from pathlib import Path

import polars as pl

AIX_DIR = Path("../data/0_aix_downloads/high_compression")
FILTERED_DIR = Path("../data/1_filtered")
TOKENIZED_DIR = Path("../data/2_tokenized")

## Stage 0: Raw Aix (downloaded from HuggingFace)


In [ ]:
# List available Aix files
aix_files = sorted(AIX_DIR.glob("*.parquet"))[:3]
print("Available Aix files:", len(list(AIX_DIR.glob("*.parquet"))))
print("Sample files:", [f.name for f in aix_files])

In [ ]:
# Load one Aix file and inspect schema
aix_sample = pl.read_parquet(aix_files[0], n_rows=5)
print("Columns:", aix_sample.columns)
print("\nSample row:")
aix_sample

## Stage 1: Filtered Games (UCI + evals + FEN)


In [ ]:
# Load filtered games
filtered_files = sorted(FILTERED_DIR.glob("*.parquet"))[:3]
print("Filtered game files:", len(list(FILTERED_DIR.glob("*.parquet"))))
print("Sample files:", [f.name for f in filtered_files])

In [ ]:
# Load sample filtered game
filtered_sample = pl.read_parquet(filtered_files[0], n_rows=5)
print("Columns:", filtered_sample.columns)
print("\nSample row:")
filtered_sample

## Stage 2: Tokenized (train/val ready)


In [ ]:
# Load tokenized datasets
pretrain_path = TOKENIZED_DIR / "pretrain.parquet"
eval_path = TOKENIZED_DIR / "eval.parquet"

pretrain_sample = pl.read_parquet(pretrain_path, n_rows=5)
eval_sample = pl.read_parquet(eval_path, n_rows=5)

print("Pretrain shape:", pl.read_parquet(pretrain_path).shape)
print("Eval shape:", pl.read_parquet(eval_path).shape)
print("\nPretrain columns:", pretrain_sample.columns)
print("\nSample tokenized row:")
pretrain_sample

## Token functions


In [ ]:
from krasnal.tokens import (
    GAME_END_ID,
    GAME_START_ID,
    ID_TO_MOVE,
)

### Opening analysis


In [ ]:
# Load filtered games and analyze openings
raw_lf = pl.scan_parquet(str(FILTERED_DIR / "*.parquet"))

openings = (
    raw_lf.select("opening")
    .collect()["opening"]
    .str.split(":")
    .list.get(0)
    .str.split(",")
    .list.get(0)
    .str.replace(r"#\d+", "")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .unique()
    .sort()
)

print("unique normalized openings:", len(openings))
for opening in openings:
    print(opening)

### Sequence statistics


In [ ]:
# length of the longest game in raw data (by move count)
raw_with_lengths = raw_lf.select(
    pl.col("uci_moves").str.split(" ").list.len().alias("move_count"),
    pl.col("uci_moves"),
).collect()

max_length = raw_with_lengths["move_count"].max()
max_idx = raw_with_lengths["move_count"].arg_max()
longest_game_moves = raw_with_lengths[max_idx, "uci_moves"]

print("longest game length (moves):", max_length)
print("example longest game:")
print(longest_game_moves)

In [ ]:
# count number of <GAME> and </GAME> tokens in pretrain parquet
flat_tokens = pl.col("token_ids").explode()
counts = pretrain_sample.select(
    flat_tokens.eq(GAME_START_ID).sum().alias("game_start_count"),
    flat_tokens.eq(GAME_END_ID).sum().alias("game_end_count"),
)

print("Tokenized column checked: token_ids")
print("<GAME> token ID:", GAME_START_ID)
print("</GAME> token ID:", GAME_END_ID)
print("total <GAME> tokens in sample:", int(counts["game_start_count"][0]))
print("total </GAME> tokens in sample:", int(counts["game_end_count"][0]))

### Tokenized game example


In [ ]:
# Example of a tokenized game with token IDs
example_game = pretrain_sample.head(1).to_dicts()[0]
token_ids = example_game["token_ids"]

print("Token IDs:", token_ids)
print("Total tokens:", len(token_ids))

In [ ]:
# Same game decoded as string tokens
tokens_decoded = [ID_TO_MOVE.get(tid, f"<{tid}>") for tid in token_ids]

print("Tokens as strings:")
for i, token in enumerate(tokens_decoded):
    print(f"  {i}: {token}")

print(f"\nTotal tokens: {len(tokens_decoded)}")